In [ ]:
#!/usr/bin/env python3
"""
Full unreduced LiH Dual-MMA TOPAZ VQE under correlated noise.
- Molecule: LiH, STO-3G, Full 12-qubit space
- Front-end: 2-layer TOPAZ/UPTE (ρ, τ)
- Ansatz: Rotating Hardware Equivalent (θ)
- Noise Evaluation: Accelerated tensordot dynamics
"""

import numpy as np
import scipy.linalg as la
import time
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from pyscf import gto, scf, fci

print("="*70)
print(" Full LiH Dual-MMA TOPAZ-style VQE (12 Qubits, FAST Tensor Dots)")
print("="*70)

N_LAYERS = 2

DEPOL_BASE = 0.005
DEPOL_CORR_LENGTH = 2.0
DEPOL_CORR_STRENGTH = 0.3

AMP_DAMP_BASE = 0.008
AMP_DAMP_CORR_LENGTH = 2.0
AMP_DAMP_CORR_STRENGTH = 0.25

PHASE_DAMP_BASE = 0.012
PHASE_DAMP_CORR_LENGTH = 2.0
PHASE_DAMP_CORR_STRENGTH = 0.3

np.random.seed(42)

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1],[1, 0]], dtype=complex)
Y = np.array([[0, -1j],[1j, 0]], dtype=complex)
Z = np.array([[1, 0],[0, -1]], dtype=complex)

def apply_single_kraus_tensor(rho_tensor, K, q, n_qubits):
    r = np.tensordot(K, rho_tensor, axes=([1], [q]))
    r = np.moveaxis(r, 0, q)
    r = np.tensordot(r, K.conj().T, axes=([n_qubits + q], [0]))
    r = np.moveaxis(r, -1, n_qubits + q)
    return r

def apply_pair_kraus_tensor(rho_tensor, K1, q1, K2, q2, n_qubits):
    r = apply_single_kraus_tensor(rho_tensor, K1, q1, n_qubits)
    r = apply_single_kraus_tensor(r, K2, q2, n_qubits)
    return r

def apply_fast_correlated_noises(rho_ideal, n_qubits):
    rho_tensor = rho_ideal.reshape([2]*(2*n_qubits))
    
    # Depolarizing
    p_single = DEPOL_BASE
    K_depol = [
        np.sqrt(1 - 3 * p_single / 4) * I2,
        np.sqrt(p_single / 4) * X,
        np.sqrt(p_single / 4) * Y,
        np.sqrt(p_single / 4) * Z,
    ]
    for q in range(n_qubits):
        r_new = np.zeros_like(rho_tensor, dtype=complex)
        for Kq in K_depol:
            r_new += apply_single_kraus_tensor(rho_tensor, Kq, q, n_qubits)
        rho_tensor = r_new
    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        p_corr = DEPOL_CORR_STRENGTH * DEPOL_BASE * np.exp(-1.0 / DEPOL_CORR_LENGTH)
        r_2q = (1.0 - p_corr) * rho_tensor
        for P1, P2 in [(X, X), (Y, Y), (Z, Z)]:
            r_temp = apply_pair_kraus_tensor(rho_tensor, P1, q1, P2, q2, n_qubits)
            r_2q += (p_corr / 3.0) * r_temp
        rho_tensor = r_2q
        
    # Amp Damp
    g_base = AMP_DAMP_BASE
    K0 = np.array([[1, 0],[0, np.sqrt(1 - g_base)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(g_base)],[0, 0]], dtype=complex)
    K_amp = [K0, K1]
    for q in range(n_qubits):
        r_new = np.zeros_like(rho_tensor, dtype=complex)
        for Kq in K_amp:
            r_new += apply_single_kraus_tensor(rho_tensor, Kq, q, n_qubits)
        rho_tensor = r_new
    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        p_corr = AMP_DAMP_CORR_STRENGTH * AMP_DAMP_BASE * np.exp(-1.0 / AMP_DAMP_CORR_LENGTH)
        r_2q = (1.0 - p_corr) * rho_tensor
        for K1a, K1b in [(K0, K0), (K0, K1), (K1, K0), (K1, K1)]:
            r_temp = apply_pair_kraus_tensor(rho_tensor, K1a, q1, K1b, q2, n_qubits)
            r_2q += (p_corr / 4.0) * r_temp
        rho_tensor = r_2q

    # Phase Damp
    l_base = PHASE_DAMP_BASE
    K0 = np.array([[1, 0],[0, np.sqrt(1 - l_base)]], dtype=complex)
    K1 = np.array([[0, 0],[0, np.sqrt(l_base)]], dtype=complex)
    K_phase = [K0, K1]
    for q in range(n_qubits):
        r_new = np.zeros_like(rho_tensor, dtype=complex)
        for Kq in K_phase:
            r_new += apply_single_kraus_tensor(rho_tensor, Kq, q, n_qubits)
        rho_tensor = r_new
    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        p_corr = PHASE_DAMP_CORR_STRENGTH * PHASE_DAMP_BASE * np.exp(-1.0 / PHASE_DAMP_CORR_LENGTH)
        r_2q = (1.0 - p_corr) * rho_tensor
        for K1a, K1b in [(K0, K0), (K0, K1), (K1, K0), (K1, K1)]:
            r_temp = apply_pair_kraus_tensor(rho_tensor, K1a, q1, K1b, q2, n_qubits)
            r_2q += (p_corr / 4.0) * r_temp
        rho_tensor = r_2q

    rho = rho_tensor.reshape((2**n_qubits, 2**n_qubits))
    tr = np.trace(rho)
    if abs(tr) > 1e-12: rho /= tr
    return rho

class MMAOptimizer:
    def __init__(self, nparams, is_angle=False):
        self.nparams = nparams
        self.movelimit = 1.5 if is_angle else 0.8
        self.gamma = 0.9
        self.L = None
        self.U = None
        self.previousparams = None
        self.previousloss = None

    def initialize_bounds(self, initialparams):
        self.L = initialparams - 1.0
        self.U = initialparams + 1.0
        self.previousparams = initialparams.copy()
        self.previousloss = None

    def solve_convex_subproblem(self, currentparams, grad):
        xnew = np.zeros_like(currentparams)
        for i in range(self.nparams):
            gi = grad[i]; xi = currentparams[i]; Li = self.L[i]; Ui = self.U[i]
            if gi > 0:
                pi = abs(gi) / max(Ui - xi, 1e-12); qi = 0.0
            else:
                pi = 0.0; qi = abs(gi) / max(xi - Li, 1e-12)
            denominator = pi * (Ui - xi + 1e-12) + qi * (xi - Li + 1e-12)
            if denominator < 1e-12: xnew[i] = xi
            else:
                numerator = pi * (Ui - xi + 1e-12) - qi * (xi - Li + 1e-12)
                xnew[i] = xi - numerator / denominator
            xnew[i] = max(xi - self.movelimit, min(xi + self.movelimit, xnew[i]))
            xnew[i] = np.clip(xnew[i], Li + 1e-6, Ui - 1e-6)
        return xnew

    def update_asymptotes(self, currentparams, currentloss):
        progressgood = True
        if self.previousloss is not None:
             progressgood = (currentloss < self.previousloss - 1e-8)
        for i in range(self.nparams):
            if progressgood:
                self.L[i] = currentparams[i] - 1.2 * self.gamma * (currentparams[i] - self.L[i])
                self.U[i] = currentparams[i] + 1.2 * self.gamma * (self.U[i] - currentparams[i])
            else:
                self.L[i] = currentparams[i] - 0.7 * self.gamma * (currentparams[i] - self.L[i])
                self.U[i] = currentparams[i] + 0.7 * self.gamma * (self.U[i] - currentparams[i])
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.previousparams = currentparams.copy()
        self.previousloss = currentloss

def finite_diff_gradient(params, objective_fn, epsilons):
    n = len(params)
    grad = np.zeros_like(params)
    for i in range(n):
        p_plus = params.copy(); p_minus = params.copy()
        p_plus[i] += epsilons[i]
        p_minus[i] -= epsilons[i]
        f_plus = objective_fn(p_plus)
        f_minus = objective_fn(p_minus)
        grad[i] = (f_plus - f_minus) / (2 * epsilons[i])
    return grad

def run_mma_optimization(initialparams, objective_fn, is_angle, epsilons, maxiterations=60, label=""):
    mma = MMAOptimizer(len(initialparams), is_angle=is_angle)
    mma.initialize_bounds(initialparams)
    currentparams = initialparams.copy()
    currentloss = objective_fn(currentparams)
    bestparams = currentparams.copy(); bestloss = currentloss; stagnation = 0

    print(f"    Initial {label} Energy: {currentloss:.6f} Ha")
    for it in range(maxiterations):
        grad = finite_diff_gradient(currentparams, objective_fn, epsilons)
        newparams = mma.solve_convex_subproblem(currentparams, grad)
        newloss = objective_fn(newparams)

        if newloss < currentloss - 1e-5:
            currentparams, currentloss, stagnation = newparams, newloss, 0
        else: stagnation += 1

        if currentloss < bestloss:
            bestloss, bestparams = currentloss, currentparams.copy()

        mma.update_asymptotes(currentparams, currentloss)
        if True:
             grad_norm = np.max(np.abs(grad))
             print(f"    Iter {it+1:3d}: Energy = {currentloss:.6f} Ha | Max Grad = {grad_norm:.6f} | Stagnation = {stagnation}")
        if stagnation >= 5:
             print(f"    Stopping MMA ({label}): Stagnation at iter {it+1}")
             break
    print(f"    MMA ({label}) complete. Best E = {bestloss:.6f}")
    return bestparams, bestloss

print("\n[1] Building Full Unreduced LiH Hamiltonian (12 Qubits)...")
driver = PySCFDriver(atom='Li 0 0 0; H 0 0 1.6', basis='sto3g', unit=DistanceUnit.ANGSTROM)
full_problem = driver.run()

# No Active Space Transformer! Matrix directly extracted from properties.
mapper = JordanWignerMapper()
qubit_op = mapper.map(full_problem.hamiltonian.second_q_op())

n_qubits = qubit_op.num_qubits
H_electronic = qubit_op.to_matrix()
nuclear_repulsion = full_problem.nuclear_repulsion_energy
H_matrix = H_electronic + nuclear_repulsion * np.eye(2**n_qubits, dtype=complex)

mol = gto.Mole()
mol.atom = 'Li 0 0 0; H 0 0 1.6'
mol.basis = 'sto-3g'
mol.build()
mf = scf.RHF(mol).run()
fci_energy = fci.FCI(mf).kernel()[0]

print(f"    Qubits (FULL Space): {n_qubits}")
print(f"    FCI (total exact):   {fci_energy:.6f} Ha")

# Generate arbitrary 1 and 2 qubit paired groupings across all 12 qubits
strings = []
for i in range(n_qubits):
    s = ['I']*n_qubits; s[i] = 'Y'; strings.append("".join(s))
    s = ['I']*n_qubits; s[i] = 'Z'; strings.append("".join(s))
for i in range(min(n_qubits - 1, 8)):
    s = ['I']*n_qubits; s[i]='X'; s[i+1]='X'; strings.append("".join(s))

PAULI_STRINGS = strings[:30] # Limit to 30 Pauli Strings for dense 4096-matrix expm performance
N_OPS = len(PAULI_STRINGS)
def pauli_string_to_matrix(pstr, n_q): return SparsePauliOp(pstr).to_matrix()
PAULI_MATS = [pauli_string_to_matrix(p, n_qubits) for p in PAULI_STRINGS]

def build_upte_from_rho_tau(rho_all, tau_all):
    dim = 2 ** n_qubits
    U = np.eye(dim, dtype=complex)
    for layer in range(len(rho_all)):
        rho, tau = rho_all[layer], tau_all[layer]
        B = sum([rho[idx] * la.expm(-1j * PAULI_MATS[idx] * tau[idx]) for idx in range(N_OPS)], np.zeros((dim, dim), dtype=complex))
        U_layer, _ = la.polar(B)
        det = np.linalg.det(U_layer)
        if abs(det) > 1e-12: U_layer *= det ** (-1.0 / dim)
        U = U_layer @ U
    return U

def build_ansatz_unitary(theta_all):
    dim = 2 ** n_qubits
    U = np.eye(dim, dtype=complex)
    for layer in range(len(theta_all)):
        theta = theta_all[layer]
        for j, H_j in enumerate(PAULI_MATS):
            U = la.expm(-1j * theta[j] * H_j) @ U
    return U

n_rho = N_LAYERS * N_OPS
n_tau = N_LAYERS * N_OPS
n_theta = N_LAYERS * N_OPS

rho_init_flat = np.random.dirichlet(np.ones(N_OPS) * 2, size=N_LAYERS).flatten()
tau_init_flat = np.random.uniform(0.1, np.pi/3, size=n_tau)
theta_init_flat = np.random.uniform(-np.pi/8, np.pi/8, size=n_theta)

current_theta, current_rho, current_tau = theta_init_flat.copy(), rho_init_flat.copy(), tau_init_flat.copy()

def build_unified_objective(rho_flat, tau_flat, theta_flat):
    rho_layers, tau_layers, theta_layers = [], [], []
    for i in range(N_LAYERS):
        r_layer = np.abs(rho_flat[i*N_OPS:(i+1)*N_OPS])
        rho_layers.append(r_layer / (r_layer.sum() + 1e-12))
        tau_layers.append(np.clip(tau_flat[i*N_OPS:(i+1)*N_OPS], 0.01, np.pi/2))
        theta_layers.append(theta_flat[i*N_OPS:(i+1)*N_OPS])

    U_topaz = build_upte_from_rho_tau(rho_layers, tau_layers)
    U_ansatz = build_ansatz_unitary(theta_layers)
    U_total = U_ansatz @ U_topaz
    
    psi0 = np.zeros(2 ** n_qubits, dtype=complex)
    psi0[195] = 1.0 # 195 translates to HF configuration across 12 STO-3G qubits
    psi_ideal = U_total @ psi0
    rho_ideal = np.outer(psi_ideal, psi_ideal.conj())
    rho_noisy = apply_fast_correlated_noises(rho_ideal, n_qubits)
    
    return np.real(np.trace(rho_noisy @ H_matrix))

print("\n[2] Phase 1 (PTE MMA): Optimizing (ρ, τ) with fixed θ(init)")
epsilons1 = np.concatenate([np.ones(n_rho) * 5e-3, np.ones(n_tau) * np.pi/16])
opt_pte_params, e1 = run_mma_optimization(
    np.concatenate([current_rho, current_tau]),
    lambda p: build_unified_objective(p[:n_rho], p[n_rho:], current_theta),
    is_angle=False, epsilons=epsilons1, maxiterations=25, label="PTE (ρ, τ)"
)
current_rho, current_tau = opt_pte_params[:n_rho], opt_pte_params[n_rho:]

print("\n[3] Phase 2 (Angle MMA): Optimizing θ with frozen PTE (ρ, τ)")
epsilons2 = np.ones(n_theta) * np.pi/16
opt_angle_params, final_energy = run_mma_optimization(
    current_theta,
    lambda p: build_unified_objective(current_rho, current_tau, p),
    is_angle=True, epsilons=epsilons2, maxiterations=25, label="Angle (θ)"
)

print("\n" + "="*70)
print(" FINAL FULL LiH UNREDUCED RESULTS")
print("="*70)
print(f"  FCI total (exact):        {fci_energy:.6f} Ha")
print(f"  Final Dual-MMA energy:    {final_energy:.6f} Ha")
print(f"  Final Error vs FCI:       {(final_energy - fci_energy)*1000:.2f} mHa")
print("="*70)


 Full LiH Dual-MMA TOPAZ-style VQE (12 Qubits, FAST Tensor Dots)

[1] Building Full Unreduced LiH Hamiltonian (12 Qubits)...
converged SCF energy = -7.86186476980865
    Qubits (FULL Space): 12
    FCI (total exact):   -7.882324 Ha

[2] Phase 1 (PTE MMA): Optimizing (ρ, τ) with fixed θ(init)
    Initial PTE (ρ, τ) Energy: -5.777045 Ha


In [ ]:
#!/usr/bin/env python3
"""
Full unreduced LiH Dual-MMA TOPAZ VQE under correlated noise.
V2-style patched version:
- Molecule: LiH, STO-3G, Full 12-qubit space
- Front-end: 2-layer TOPAZ/U-PTE (rho, tau)
- Ansatz: Rotating Hardware Equivalent (theta)
- Noise Evaluation: Accelerated tensor-dot dynamics
- MMA v2 additions:
    * gradient momentum smoothing
    * adaptive move limits
    * gamma = 0.5 asymptote management
    * stabilized polar projection / unitary regularization
    * optional heuristic virtual distillation post-processing
"""

import numpy as np
import scipy.linalg as la
import time
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from pyscf import gto, scf, fci


print("=" * 70)
print(" Full LiH Dual-MMA TOPAZ-style VQE (12 Qubits, FAST Tensor Dots, V2)")
print("=" * 70)

# ---------------------------------------------------------------------
# Global controls
# ---------------------------------------------------------------------
N_LAYERS = 2

DEPOL_BASE = 0.005
DEPOL_CORR_LENGTH = 2.0
DEPOL_CORR_STRENGTH = 0.3

AMP_DAMP_BASE = 0.008
AMP_DAMP_CORR_LENGTH = 2.0
AMP_DAMP_CORR_STRENGTH = 0.25

PHASE_DAMP_BASE = 0.012
PHASE_DAMP_CORR_LENGTH = 2.0
PHASE_DAMP_CORR_STRENGTH = 0.3

USE_HEURISTIC_VD = True
VD_COPIES = 2

np.random.seed(42)

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)


# ---------------------------------------------------------------------
# Tensor-dot noise engine
# ---------------------------------------------------------------------
def apply_single_kraus_tensor(rho_tensor, K, q, n_qubits):
    r = np.tensordot(K, rho_tensor, axes=([1], [q]))
    r = np.moveaxis(r, 0, q)
    r = np.tensordot(r, K.conj().T, axes=([n_qubits + q], [0]))
    r = np.moveaxis(r, -1, n_qubits + q)
    return r


def apply_pair_kraus_tensor(rho_tensor, K1, q1, K2, q2, n_qubits):
    r = apply_single_kraus_tensor(rho_tensor, K1, q1, n_qubits)
    r = apply_single_kraus_tensor(r, K2, q2, n_qubits)
    return r


def apply_fast_correlated_noises(rho_ideal, n_qubits):
    rho_tensor = rho_ideal.reshape([2] * (2 * n_qubits))

    # Depolarizing
    p_single = DEPOL_BASE
    K_depol = [
        np.sqrt(1 - 3 * p_single / 4) * I2,
        np.sqrt(p_single / 4) * X,
        np.sqrt(p_single / 4) * Y,
        np.sqrt(p_single / 4) * Z,
    ]
    for q in range(n_qubits):
        r_new = np.zeros_like(rho_tensor, dtype=complex)
        for Kq in K_depol:
            r_new += apply_single_kraus_tensor(rho_tensor, Kq, q, n_qubits)
        rho_tensor = r_new

    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        p_corr = DEPOL_CORR_STRENGTH * DEPOL_BASE * np.exp(-1.0 / DEPOL_CORR_LENGTH)
        r_2q = (1.0 - p_corr) * rho_tensor
        for P1, P2 in [(X, X), (Y, Y), (Z, Z)]:
            r_temp = apply_pair_kraus_tensor(rho_tensor, P1, q1, P2, q2, n_qubits)
            r_2q += (p_corr / 3.0) * r_temp
        rho_tensor = r_2q

    # Amplitude damping
    g_base = AMP_DAMP_BASE
    K0_amp = np.array([[1, 0], [0, np.sqrt(1 - g_base)]], dtype=complex)
    K1_amp = np.array([[0, np.sqrt(g_base)], [0, 0]], dtype=complex)
    K_amp = [K0_amp, K1_amp]

    for q in range(n_qubits):
        r_new = np.zeros_like(rho_tensor, dtype=complex)
        for Kq in K_amp:
            r_new += apply_single_kraus_tensor(rho_tensor, Kq, q, n_qubits)
        rho_tensor = r_new

    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        p_corr = AMP_DAMP_CORR_STRENGTH * AMP_DAMP_BASE * np.exp(-1.0 / AMP_DAMP_CORR_LENGTH)
        r_2q = (1.0 - p_corr) * rho_tensor
        for A, B in [(K0_amp, K0_amp), (K0_amp, K1_amp), (K1_amp, K0_amp), (K1_amp, K1_amp)]:
            r_temp = apply_pair_kraus_tensor(rho_tensor, A, q1, B, q2, n_qubits)
            r_2q += (p_corr / 4.0) * r_temp
        rho_tensor = r_2q

    # Phase damping
    l_base = PHASE_DAMP_BASE
    K0_phase = np.array([[1, 0], [0, np.sqrt(1 - l_base)]], dtype=complex)
    K1_phase = np.array([[0, 0], [0, np.sqrt(l_base)]], dtype=complex)
    K_phase = [K0_phase, K1_phase]

    for q in range(n_qubits):
        r_new = np.zeros_like(rho_tensor, dtype=complex)
        for Kq in K_phase:
            r_new += apply_single_kraus_tensor(rho_tensor, Kq, q, n_qubits)
        rho_tensor = r_new

    for q1 in range(n_qubits - 1):
        q2 = q1 + 1
        p_corr = PHASE_DAMP_CORR_STRENGTH * PHASE_DAMP_BASE * np.exp(-1.0 / PHASE_DAMP_CORR_LENGTH)
        r_2q = (1.0 - p_corr) * rho_tensor
        for A, B in [(K0_phase, K0_phase), (K0_phase, K1_phase), (K1_phase, K0_phase), (K1_phase, K1_phase)]:
            r_temp = apply_pair_kraus_tensor(rho_tensor, A, q1, B, q2, n_qubits)
            r_2q += (p_corr / 4.0) * r_temp
        rho_tensor = r_2q

    rho = rho_tensor.reshape((2 ** n_qubits, 2 ** n_qubits))
    tr = np.trace(rho)
    if abs(tr) > 1e-12:
        rho /= tr
    return rho


# ---------------------------------------------------------------------
# Heuristic virtual distillation
# ---------------------------------------------------------------------
def apply_virtual_distillation_heuristic(rho, copies=2):
    """
    Heuristic post-processing only.
    Repeated matrix-power normalization acts like a simplified
    signal-amplification distillation proxy.
    """
    if copies < 2:
        return rho

    rho_vd = rho.copy()
    for _ in range(copies - 1):
        rho_vd = rho_vd @ rho

    tr = np.trace(rho_vd)
    if abs(tr) > 1e-12:
        rho_vd /= tr
    return rho_vd


# ---------------------------------------------------------------------
# MMA v2 optimizer
# ---------------------------------------------------------------------
class MMAOptimizer:
    def __init__(self, nparams, is_angle=False):
        self.nparams = nparams
        self.is_angle = is_angle

        # v2-style asymptote control
        self.gamma = 0.5

        # v2-style move-limit adaptation
        self.max_movelimit = 1.5 if is_angle else 0.6
        self.min_movelimit = 0.15
        self.movelimit = 0.6 if is_angle else 0.4

        # v2-style gradient smoothing
        self.grad_momentum = 0.4
        self.prev_grad = np.zeros(nparams, dtype=float)

        self.L = None
        self.U = None
        self.previousparams = None
        self.previousloss = None

    def initialize_bounds(self, initialparams):
        self.L = initialparams - 1.0
        self.U = initialparams + 1.0
        self.previousparams = initialparams.copy()
        self.previousloss = None

    def smooth_gradient(self, raw_grad):
        grad = self.grad_momentum * raw_grad + (1.0 - self.grad_momentum) * self.prev_grad
        self.prev_grad = grad.copy()
        return grad

    def update_movelimit(self, improvement):
        # v2-style improvement logic:
        # strong > 0.01, moderate 0.001..0.01, weak < 0.001
        if improvement > 1e-2:
            self.movelimit = min(self.movelimit * 1.15, self.max_movelimit)
        elif improvement > 1e-3:
            self.movelimit = self.movelimit
        else:
            self.movelimit = max(self.movelimit * 0.85, self.min_movelimit)

    def solve_convex_subproblem(self, currentparams, grad):
        xnew = np.zeros_like(currentparams)

        for i in range(self.nparams):
            gi = grad[i]
            xi = currentparams[i]
            Li = self.L[i]
            Ui = self.U[i]

            if gi > 0:
                pi = abs(gi) / max(Ui - xi, 1e-12)
                qi = 0.0
            else:
                pi = 0.0
                qi = abs(gi) / max(xi - Li, 1e-12)

            denominator = pi * (Ui - xi + 1e-12) + qi * (xi - Li + 1e-12)
            if denominator < 1e-12:
                xnew[i] = xi
            else:
                numerator = pi * (Ui - xi + 1e-12) - qi * (xi - Li + 1e-12)
                xnew[i] = xi - numerator / denominator

            xnew[i] = max(xi - self.movelimit, min(xi + self.movelimit, xnew[i]))
            xnew[i] = np.clip(xnew[i], Li + 1e-6, Ui - 1e-6)

        return xnew

    def update_asymptotes(self, currentparams, currentloss):
        progressgood = True
        if self.previousloss is not None:
            progressgood = (currentloss < self.previousloss - 1e-8)

        for i in range(self.nparams):
            if progressgood:
                self.L[i] = currentparams[i] - 1.2 * self.gamma * (currentparams[i] - self.L[i])
                self.U[i] = currentparams[i] + 1.2 * self.gamma * (self.U[i] - currentparams[i])
            else:
                self.L[i] = currentparams[i] - 1.0 * self.gamma * (currentparams[i] - self.L[i])
                self.U[i] = currentparams[i] + 1.0 * self.gamma * (self.U[i] - currentparams[i])

        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.previousparams = currentparams.copy()
        self.previousloss = currentloss


# ---------------------------------------------------------------------
# Numerical gradients
# ---------------------------------------------------------------------
def finite_diff_gradient(params, objective_fn, epsilons):
    n = len(params)
    grad = np.zeros_like(params, dtype=float)

    for i in range(n):
        p_plus = params.copy()
        p_minus = params.copy()
        p_plus[i] += epsilons[i]
        p_minus[i] -= epsilons[i]
        f_plus = objective_fn(p_plus)
        f_minus = objective_fn(p_minus)
        grad[i] = (f_plus - f_minus) / (2 * epsilons[i])

    return grad


def run_mma_optimization(initialparams, objective_fn, is_angle, epsilons, maxiterations=60, label=""):
    mma = MMAOptimizer(len(initialparams), is_angle=is_angle)
    mma.initialize_bounds(initialparams)

    currentparams = initialparams.copy()
    currentloss = objective_fn(currentparams)
    bestparams = currentparams.copy()
    bestloss = currentloss
    stagnation = 0

    print(f"    Initial {label} Energy: {currentloss:.6f} Ha")

    for it in range(maxiterations):
        raw_grad = finite_diff_gradient(currentparams, objective_fn, epsilons)
        grad = mma.smooth_gradient(raw_grad)

        newparams = mma.solve_convex_subproblem(currentparams, grad)
        newloss = objective_fn(newparams)
        improvement = currentloss - newloss

        mma.update_movelimit(max(improvement, 0.0))

        if newloss < currentloss - 1e-5:
            currentparams, currentloss = newparams, newloss
            stagnation = 0
        else:
            stagnation += 1

        if currentloss < bestloss:
            bestloss = currentloss
            bestparams = currentparams.copy()

        mma.update_asymptotes(currentparams, currentloss)

        grad_norm = np.max(np.abs(grad))
        print(
            f"    Iter {it+1:3d}: Energy = {currentloss:.6f} Ha | "
            f"Max Grad = {grad_norm:.6f} | Move = {mma.movelimit:.4f} | "
            f"Stagnation = {stagnation}"
        )

        if stagnation >= 5:
            print(f"    Stopping MMA ({label}): Stagnation at iter {it+1}")
            break

    print(f"    MMA ({label}) complete. Best E = {bestloss:.6f}")
    return bestparams, bestloss


# ---------------------------------------------------------------------
# Hamiltonian build
# ---------------------------------------------------------------------
print("\n[1] Building Full Unreduced LiH Hamiltonian (12 Qubits)...")
driver = PySCFDriver(atom='Li 0 0 0; H 0 0 1.6', basis='sto3g', unit=DistanceUnit.ANGSTROM)
full_problem = driver.run()

mapper = JordanWignerMapper()
qubit_op = mapper.map(full_problem.hamiltonian.second_q_op())

n_qubits = qubit_op.num_qubits
H_electronic = qubit_op.to_matrix()
nuclear_repulsion = full_problem.nuclear_repulsion_energy
H_matrix = H_electronic + nuclear_repulsion * np.eye(2 ** n_qubits, dtype=complex)

mol = gto.Mole()
mol.atom = 'Li 0 0 0; H 0 0 1.6'
mol.basis = 'sto-3g'
mol.build()
mf = scf.RHF(mol).run()
fci_energy = fci.FCI(mf).kernel()[0]

print(f"    Qubits (FULL Space): {n_qubits}")
print(f"    FCI (total exact):   {fci_energy:.6f} Ha")


# ---------------------------------------------------------------------
# Pauli basis selection
# ---------------------------------------------------------------------
strings = []
for i in range(n_qubits):
    s = ['I'] * n_qubits
    s[i] = 'Y'
    strings.append("".join(s))

    s = ['I'] * n_qubits
    s[i] = 'Z'
    strings.append("".join(s))

for i in range(min(n_qubits - 1, 8)):
    s = ['I'] * n_qubits
    s[i] = 'X'
    s[i + 1] = 'X'
    strings.append("".join(s))

PAULI_STRINGS = strings[:30]  # cap for dense expm cost
N_OPS = len(PAULI_STRINGS)


def pauli_string_to_matrix(pstr):
    return SparsePauliOp(pstr).to_matrix()


PAULI_MATS = [pauli_string_to_matrix(p) for p in PAULI_STRINGS]


# ---------------------------------------------------------------------
# U-PTE / ansatz builders
# ---------------------------------------------------------------------
def stable_polar_unitary(B, eps=1e-10):
    """
    Stabilized polar projection:
        U = B (B^\dagger B + eps I)^(-1/2)
    with SU(d) determinant normalization.
    """
    dim = B.shape[0]
    M = B.conj().T @ B + eps * np.eye(dim, dtype=complex)
    M_inv_sqrt = la.fractional_matrix_power(M, -0.5)
    U = B @ M_inv_sqrt

    det = np.linalg.det(U)
    if abs(det) > 1e-12:
        U *= det ** (-1.0 / dim)

    return U


def build_upte_from_rho_tau(rho_all, tau_all):
    dim = 2 ** n_qubits
    U = np.eye(dim, dtype=complex)

    for layer in range(len(rho_all)):
        rho = rho_all[layer]
        tau = tau_all[layer]

        B = np.zeros((dim, dim), dtype=complex)
        for idx in range(N_OPS):
            B += rho[idx] * la.expm(-1j * PAULI_MATS[idx] * tau[idx])

        U_layer = stable_polar_unitary(B)
        U = U_layer @ U

    return U


def build_ansatz_unitary(theta_all):
    dim = 2 ** n_qubits
    U = np.eye(dim, dtype=complex)

    for layer in range(len(theta_all)):
        theta = theta_all[layer]
        for j, H_j in enumerate(PAULI_MATS):
            U = la.expm(-1j * theta[j] * H_j) @ U

    return U


# ---------------------------------------------------------------------
# Parameter initialization
# ---------------------------------------------------------------------
n_rho = N_LAYERS * N_OPS
n_tau = N_LAYERS * N_OPS
n_theta = N_LAYERS * N_OPS

rho_init_flat = np.random.dirichlet(np.ones(N_OPS) * 2, size=N_LAYERS).flatten()
tau_init_flat = np.random.uniform(0.1, np.pi / 3, size=n_tau)
theta_init_flat = np.random.uniform(-np.pi / 8, np.pi / 8, size=n_theta)

current_theta = theta_init_flat.copy()
current_rho = rho_init_flat.copy()
current_tau = tau_init_flat.copy()


# ---------------------------------------------------------------------
# Unified objective
# ---------------------------------------------------------------------
def build_unified_state_and_density(rho_flat, tau_flat, theta_flat):
    rho_layers, tau_layers, theta_layers = [], [], []

    for i in range(N_LAYERS):
        r_layer = np.abs(rho_flat[i * N_OPS:(i + 1) * N_OPS])
        rho_layers.append(r_layer / (r_layer.sum() + 1e-12))
        tau_layers.append(np.clip(tau_flat[i * N_OPS:(i + 1) * N_OPS], 0.01, np.pi / 2))
        theta_layers.append(theta_flat[i * N_OPS:(i + 1) * N_OPS])

    U_topaz = build_upte_from_rho_tau(rho_layers, tau_layers)
    U_ansatz = build_ansatz_unitary(theta_layers)
    U_total = U_ansatz @ U_topaz

    psi0 = np.zeros(2 ** n_qubits, dtype=complex)
    psi0[195] = 1.0  # HF reference determinant index for this setup

    psi_ideal = U_total @ psi0
    rho_ideal = np.outer(psi_ideal, psi_ideal.conj())
    rho_noisy = apply_fast_correlated_noises(rho_ideal, n_qubits)

    return psi_ideal, rho_ideal, rho_noisy


def build_unified_objective(rho_flat, tau_flat, theta_flat):
    _, _, rho_noisy = build_unified_state_and_density(rho_flat, tau_flat, theta_flat)
    return np.real(np.trace(rho_noisy @ H_matrix))


# ---------------------------------------------------------------------
# Phase 1: PTE MMA
# ---------------------------------------------------------------------
print("\n[2] Phase 1 (PTE MMA): Optimizing (rho, tau) with fixed theta(init)")
epsilons1 = np.concatenate([
    np.ones(n_rho) * 5e-3,
    np.ones(n_tau) * (np.pi / 16)
])

opt_pte_params, e1 = run_mma_optimization(
    np.concatenate([current_rho, current_tau]),
    lambda p: build_unified_objective(p[:n_rho], p[n_rho:], current_theta),
    is_angle=False,
    epsilons=epsilons1,
    maxiterations=25,
    label="PTE (rho, tau)"
)

current_rho = opt_pte_params[:n_rho]
current_tau = opt_pte_params[n_rho:]


# ---------------------------------------------------------------------
# Phase 2: Angle MMA
# ---------------------------------------------------------------------
print("\n[3] Phase 2 (Angle MMA): Optimizing theta with frozen PTE (rho, tau)")
epsilons2 = np.ones(n_theta) * (np.pi / 16)

opt_angle_params, final_energy = run_mma_optimization(
    current_theta,
    lambda p: build_unified_objective(current_rho, current_tau, p),
    is_angle=True,
    epsilons=epsilons2,
    maxiterations=25,
    label="Angle (theta)"
)

current_theta = opt_angle_params.copy()


# ---------------------------------------------------------------------
# Final evaluation
# ---------------------------------------------------------------------
print("\n[4] Final evaluation...")
psi_ideal, rho_ideal, rho_noisy = build_unified_state_and_density(current_rho, current_tau, current_theta)
final_energy = np.real(np.trace(rho_noisy @ H_matrix))

if USE_HEURISTIC_VD:
    rho_vd = apply_virtual_distillation_heuristic(rho_noisy, copies=VD_COPIES)
    vd_energy = np.real(np.trace(rho_vd @ H_matrix))
else:
    vd_energy = None

print("\n" + "=" * 70)
print(" FINAL FULL LiH UNREDUCED RESULTS (TOPAZ V2-STYLE PATCH)")
print("=" * 70)
print(f"  FCI total (exact):              {fci_energy:.6f} Ha")
print(f"  Final Dual-MMA energy:          {final_energy:.6f} Ha")
print(f"  Final Error vs FCI:             {(final_energy - fci_energy) * 1000:.2f} mHa")
if vd_energy is not None:
    print(f"  Heuristic VD energy ({VD_COPIES} copies):   {vd_energy:.6f} Ha")
    print(f"  Heuristic VD Error vs FCI:      {(vd_energy - fci_energy) * 1000:.2f} mHa")
print("=" * 70)

<>:396: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:396: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/var/folders/04/fy49_m1s2nv5q_z8q97vkcvr0000gn/T/ipykernel_67833/170385470.py:396: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  U = B (B^\dagger B + eps I)^(-1/2)


 Full LiH Dual-MMA TOPAZ-style VQE (12 Qubits, FAST Tensor Dots, V2)

[1] Building Full Unreduced LiH Hamiltonian (12 Qubits)...
converged SCF energy = -7.86186476980865
    Qubits (FULL Space): 12
    FCI (total exact):   -7.882324 Ha

[2] Phase 1 (PTE MMA): Optimizing (rho, tau) with fixed theta(init)
    Initial PTE (rho, tau) Energy: -5.777045 Ha
